# Этап 3: Конструирование признаков (Feature Engineering)

## Описание файла
Скрипт выполняет профилирование выделенных на предыдущем этапе кластеров (сущностей). Для последующего применения алгоритмов машинного обучения транзакционная история трансформируется в набор числовых поведенческих метрик.

**Извлекаемые признаки (Features):**
* `Tx_Count` — суммарная транзакционная активность (частота переводов).
* `Address_Count` — размер инфраструктуры кошелька (количество адресов).
* `Total_Received` / `Total_Sent` — финансовый оборот сущности.
* `Balance` — текущий остаток средств (в рамках исследуемого временного окна).

**Цель:** Подготовить структурированный датасет для кластеризации поведения (Behavioral Clustering).

**Входные данные:** `../data/blockchain_final_entities.csv`  
**Выходные данные:** `../data/entity_features.csv`

In [1]:
import pandas as pd
import numpy as np

def build_entity_features(csv_path):
    print("1. Загрузка финального датасета с кластерами...")
    df = pd.read_csv(csv_path)

    df = df.dropna(subset=['Entity_ID'])
    
    print("2. Сбор статистики по каждой сущности...")
    
    received = df[df['Type'] == 'OUTPUT'].groupby('Entity_ID')['Amount_BTC'].sum().reset_index()
    received.columns = ['Entity_ID', 'Total_Received']
    
    sent = df[df['Type'] == 'INPUT'].groupby('Entity_ID')['Amount_BTC'].sum().reset_index()
    sent.columns = ['Entity_ID', 'Total_Sent']
    
    tx_counts = df.groupby('Entity_ID')['TxID'].nunique().reset_index()
    tx_counts.columns = ['Entity_ID', 'Tx_Count']
    
    addr_counts = df.groupby('Entity_ID')['Address'].nunique().reset_index()
    addr_counts.columns = ['Entity_ID', 'Address_Count']
    
    print("3. Объединение досье в единую таблицу...")
    features_df = tx_counts.merge(addr_counts, on='Entity_ID', how='outer')
    features_df = features_df.merge(received, on='Entity_ID', how='left')
    features_df = features_df.merge(sent, on='Entity_ID', how='left')
    
    features_df = features_df.fillna(0)
    
    features_df['Balance'] = features_df['Total_Received'] - features_df['Total_Sent']
    
    features_df = features_df.sort_values(by='Tx_Count', ascending=False)
    
    print("4. Сохранение признаков (Features)...")
    output_file = "entity_features.csv" 
    features_df.to_csv(output_file, index=False)
    print(f"Готово! Досье сохранены в {output_file}")
    
    return features_df

if __name__ == "__main__":
    
    input_file = "blockchain_final_entities.csv" 
    
    df_features = build_entity_features(input_file)
    
    print("\nТоп-5 самых активных сущностей (потенциальные биржи/сервисы):")
    print(df_features.head())
    
    print(f"\nВсего собрано досье на {len(df_features)} уникальных сущностей.")

1. Загрузка финального датасета с кластерами...
2. Сбор статистики по каждой сущности...
3. Объединение досье в единую таблицу...
4. Сохранение признаков (Features)...
Готово! Досье сохранены в entity_features.csv

Топ-5 самых активных сущностей (потенциальные биржи/сервисы):
      Entity_ID  Tx_Count  Address_Count  Total_Received   Total_Sent  \
753  Cluster_99       382             73     1532.480598  1530.953268   
743   Cluster_9        78              1        1.458251     1.433140   
46   Cluster_14        38              1        0.829862     1.162296   
221   Cluster_3        31              9       28.990979    32.914631   
123  Cluster_21        15            323        1.370012    30.302013   

       Balance  
753   1.527330  
743   0.025112  
46   -0.332434  
221  -3.923652  
123 -28.932001  

Всего собрано досье на 754 уникальных сущностей.
